# 10. All Stage 10 tests, with Holm adjusted p values

**What this notebook does.** It reads the tests saved by notebooks 06 to 09, puts them in one table and adds the Holm adjusted p values across the primary tests. It fits no model and computes no new result.

**Exploratory tests.** Two checks were added after the Stage 10 results were seen (`ANALYSIS_PLAN.md`, "Exploratory, added after Stage 10 results"). Their tests are in the table with the role `exploratory`. They are not adjusted, and they do not change the Holm values of the primary tests.

**Plan.** `ANALYSIS_PLAN.md` section 10.1 (reporting rules) and 10.7.

**The 8 primary tests (the Holm family), fixed in the plan:**
- the coefficient d on the 24h move (notebook 06);
- the mean change in a team's price after a key player is ruled Out (notebook 07);
- the slope by group interaction for liquidity, season half and rest (notebook 08);
- the calibration LR test, the popularity coefficient c and the return of the opposite rule, all on 2026-27 (notebook 09).

**Holm.** Sort the 8 primary p values from smallest to largest. The adjusted value of the i-th is the largest of (8 − j + 1) · p(j) over j ≤ i, capped at 1. A primary test is significant when its adjusted p value is below 0.05. Secondary tests keep their raw p value and are not adjusted.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from IPython.display import display

import all_tests
from csv_io import read_csv, write_csv

parts = [read_csv(all_tests.part_path(part)) for part in all_tests.PARTS]
exploratory_parts = [read_csv(all_tests.part_path(name)) for name in all_tests.EXPLORATORY_FILES]
combined = all_tests.combine(parts + exploratory_parts)
write_csv(combined, all_tests.ALL_TESTS_PATH)
print(f"{len(combined)} tests: {(combined['role'] == 'primary').sum()} primary, "
      f"{(combined['role'] == 'secondary').sum()} secondary, "
      f"{(combined['role'] == 'exploratory').sum()} exploratory")

66 tests: 8 primary, 47 secondary, 11 exploratory


## 1. The primary tests

Sorted by raw p value. `p_holm` is the adjusted value.

In [2]:
primary = combined[combined["role"] == "primary"].sort_values("p_raw")
display(primary[["part", "test_id", "n", "unit", "estimate", "ci_lo", "ci_hi", "statistic",
                 "statistic_name", "df", "p_raw", "p_holm", "significant_holm"]].round(4))

,part,test_id,n,unit,estimate,ci_lo,ci_hi,statistic,statistic_name,df,p_raw,p_holm,significant_holm
12,information_arrival,injury_event_mean,302,events,-0.0089,-0.0113,-0.0067,NaN,NaN,NaN,0.0010,0.0080,1
32,subgroups,subgroup_season_half_slope,1229,games,0.4451,0.1421,0.7480,2.879675,z,<NA>,0.0040,0.0279,1
53,replication,replication_opposite_return,72,trades,0.0936,-0.0268,0.2031,NaN,NaN,<NA>,0.1219,0.7316,0
2,line_movement,line_move_24h_d,1222,games,0.3883,-0.1136,0.8903,1.516456,z,NaN,0.1294,0.7316,0
46,replication,replication_lr,248,games,NaN,NaN,NaN,2.346059,LR,2,0.3094,1.0000,0
18,subgroups,subgroup_liquidity_slope,1229,games,NaN,NaN,NaN,2.340693,chi2,2,0.3103,1.0000,0
52,replication,replication_popularity_c,248,games,0.2451,-0.2941,0.7842,0.89091,z,<NA>,0.3730,1.0000,0
39,subgroups,subgroup_rest_slope,1229,games,-0.1314,-0.4225,0.1596,-0.885119,z,<NA>,0.3761,1.0000,0


**Significant after the Holm adjustment (2 of 8).**
- **Injury report event** (notebook 07): the team's price falls 0.89 points on average when a key player is ruled Out. Raw p = 0.001, Holm 0.008.
- **Season half** (notebook 08): the calibration slope is 0.445 higher in the late half of 2025-26 than in the early half. Raw p = 0.004, Holm 0.028.

**Not significant (6 of 8).**
- Opposite rule return on 2026-27: +9.4%. Raw p = 0.122, Holm 0.73.
- Line movement, d on the 24h move: 0.388. Raw p = 0.129, Holm 0.73.
- Calibration LR test on 2026-27: raw p = 0.309, Holm 1.
- Liquidity, slope by group: raw p = 0.310, Holm 1.
- Popularity c on 2026-27: 0.245. Raw p = 0.373, Holm 1.
- Rest, slope by group: raw p = 0.376, Holm 1.

A test that is not significant here does not show that the effect is absent. The intervals in the table say how large an effect each test could have missed, and for the 2026-27 tests they are wide.

## 2. The secondary tests

Raw p values, not adjusted. Those below 0.05 are listed first.

In [3]:
secondary = combined[combined["role"] == "secondary"].sort_values("p_raw")
print(f"{len(secondary)} secondary tests, {(secondary['p_raw'] < 0.05).sum()} with p below 0.05")
display(secondary[secondary["p_raw"] < 0.05][["part", "test_id", "description", "n", "estimate",
                                              "ci_lo", "ci_hi", "p_raw"]].round(4))

47 secondary tests, 8 with p below 0.05


,part,test_id,description,n,estimate,ci_lo,ci_hi,p_raw
17,information_arrival,injury_anticipation_minus_placebo,mean of anticipation change minus its placebo,301,-0.0058,-0.0085,-0.0034,0.0010
14,information_arrival,injury_event_minus_placebo,mean of event change minus placebo change,302,-0.0089,-0.0115,-0.0066,0.0010
15,information_arrival,injury_anticipation_mean,"mean change of the affected team's price, 60 t...",302,-0.0063,-0.0086,-0.0042,0.0010
38,subgroups,subgroup_season_half_brier_late_minus_early,season half split: Brier score of late minus e...,1229,-0.0448,-0.0639,-0.0250,0.0010
49,replication,replication_brier_diff,2026-27 Brier score minus the constant baseline,248,-0.0563,-0.0771,-0.0343,0.0010
37,subgroups,subgroup_season_half_late_b,"season half split, late group: b",618,1.2612,1.0610,1.4615,0.0105
31,subgroups,subgroup_liquidity_brier_high_minus_middle,liquidity split: Brier score of high minus middle,1229,0.0257,0.0056,0.0471,0.0120
54,replication,replication_opposite_benchmark_return,opposite rule benchmark: return per dollar staked,248,-0.0969,-0.1933,-0.0109,0.0310


8 of the 47 secondary tests have a raw p value below 0.05. With 47 tests and no adjustment, two or three would be expected by chance alone, so each of these is weaker evidence than a primary test.

Six of the eight restate something already seen:
- three belong to the injury event study (the change minus its placebo, and the two anticipation tests);
- two belong to the season-half split (the late-half slope against 1, and the Brier difference between the halves);
- one says the 2026-27 prices beat a constant forecast.

Two stand alone: the Brier difference between the high and middle volume groups, and the loss of the 2026-27 benchmark (buying the more popular team in every game).

## 3. The exploratory tests

Added after the Stage 10 results were seen. Raw p values, not adjusted. The injury event rows come from notebook 07 and the slope rows from notebook 08, where they are explained.

In [4]:
exploratory = combined[combined["role"] == "exploratory"]
display(exploratory[["part", "test_id", "n", "unit", "estimate", "ci_lo", "ci_hi", "interval_type",
                     "p_raw"]].round(4))

,part,test_id,n,unit,estimate,ci_lo,ci_hi,interval_type,p_raw
55,information_arrival,exploratory_injury_questionable_main,193,events,-0.0128,-0.0163,-0.0097,bootstrap percentile,0.0010
56,information_arrival,exploratory_injury_questionable_anticipation,193,events,-0.0082,-0.0117,-0.0053,bootstrap percentile,0.0010
57,information_arrival,exploratory_injury_doubtful_main,72,events,-0.0004,-0.0022,0.0014,bootstrap percentile,0.6167
58,information_arrival,exploratory_injury_doubtful_anticipation,72,events,-0.0007,-0.0030,0.0016,bootstrap percentile,0.5337
59,information_arrival,exploratory_injury_not_listed_main,27,events,-0.0063,-0.0132,-0.0015,bootstrap percentile,0.0040
60,information_arrival,exploratory_injury_not_listed_anticipation,27,events,-0.0059,-0.0152,0.0011,bootstrap percentile,0.1279
61,subgroups,exploratory_slope_2025_10_2025_11,299,games,1.1260,0.8119,1.4402,clustered Wald,0.4317
62,subgroups,exploratory_slope_2025_12,197,games,0.5111,0.2389,0.7833,clustered Wald,0.0004
63,subgroups,exploratory_slope_2026_01,232,games,0.8336,0.5205,1.1468,clustered Wald,0.2977
64,subgroups,exploratory_slope_2026_02,166,games,1.3893,0.9103,1.8683,clustered Wald,0.1112


11 exploratory tests, none adjusted.

- **Injury events by previous status (6 tests).** The price falls when a Questionable player is ruled Out (-1.28 points in the main window, -0.82 in the hour before, both p = 0.001). It does not move when a Doubtful player is ruled Out (p = 0.62 and 0.53). For players who were not listed the main-window mean is -0.63 points (p = 0.004), on 27 events.
- **Slope by month (5 tests).** December has b = 0.51 (p = 0.0004) and March to April b = 1.30 (p = 0.024). The other three groups are not distinguishable from 1.

These were chosen after the planned results were seen. They help to read the two significant primary tests. They are not confirmation of them.

## 4. Every test

In [5]:
with pd.option_context("display.max_rows", 200):
    display(combined[["part", "test_id", "role", "n", "unit", "estimate", "ci_lo", "ci_hi",
                      "boot_lo", "boot_hi", "statistic", "df", "p_raw", "p_holm"]].round(4))

,part,test_id,role,n,unit,estimate,ci_lo,ci_hi,boot_lo,boot_hi,statistic,df,p_raw,p_holm
0,line_movement,line_move_24h_a,secondary,1222,games,0.0164,-0.0746,0.1074,-0.101116,0.139717,0.352945,NaN,0.7241,NaN
1,line_movement,line_move_24h_b,secondary,1222,games,1.0052,0.8775,1.1329,0.877936,1.16368,0.079982,NaN,0.9363,NaN
2,line_movement,line_move_24h_d,primary,1222,games,0.3883,-0.1136,0.8903,-0.177291,0.958062,1.516456,NaN,0.1294,0.7316
3,line_movement,line_move_6h_a,secondary,1222,games,0.0188,-0.0710,0.1086,-0.096516,0.141812,0.410644,NaN,0.6813,NaN
4,line_movement,line_move_6h_b,secondary,1222,games,1.0308,0.9081,1.1534,0.903346,1.186536,0.491466,NaN,0.6231,NaN
5,line_movement,line_move_6h_d,secondary,1222,games,0.3431,-0.6842,1.3703,-0.521564,1.315075,0.654549,NaN,0.5128,NaN
6,line_movement,line_move_1h_a,secondary,1222,games,0.0166,-0.0728,0.1061,-0.100907,0.139094,0.364224,NaN,0.7157,NaN
7,line_movement,line_move_1h_b,secondary,1222,games,1.0342,0.9124,1.1559,0.907115,1.196295,0.55017,NaN,0.5822,NaN
8,line_movement,line_move_1h_d,secondary,1222,games,0.0950,-1.6157,1.8056,-1.712553,2.093389,0.108812,NaN,0.9134,NaN
9,line_movement,line_move_24h_spread_filter_a,secondary,1165,games,0.0079,-0.0887,0.1045,-0.113917,0.136327,0.16052,NaN,0.8725,NaN
